In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ============================================================
# Part 1: 2002-2021 -- daily maximum from local hourly files sampled 3-hourly
# ============================================================
import xarray as xr
import numpy as np
from pathlib import Path

SRC = Path(rf"{DATA_ROOT}/AIS_processing/weather_data/wave")   # era5_swh_{year}.nc, hourly
OUT = Path(rf"{DATA_ROOT}/era5_swh_dailymax")
OUT.mkdir(parents=True, exist_ok=True)

for year in range(2002, 2022):
    out_f = OUT / f"{year}_SWH_dailymax.nc"
    if out_f.exists():
        print(f"[SKIP] {year}")
        continue

    ds = xr.open_dataset(SRC / f"era5_swh_{year}.nc", chunks={"valid_time": 240})
    da = ds["swh"]

    # Key: take 00,03,...,21 UTC to align with WW3 3-hourly output times
    da = da.sel(valid_time=da.valid_time.dt.hour.isin([0, 3, 6, 9, 12, 15, 18, 21]))
    assert len(da.valid_time) in (2920, 2928), f"{year}: unexpected number of time steps {len(da.valid_time)}"

    dmax = da.resample(valid_time="1D").max()

    # Standardise names/order: time / lat / lon, latitude ascending (as in WW3)
    dmax = (dmax.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
                .sortby("lat")
                .rename("swh"))
    dmax = dmax.drop_vars([v for v in ("number", "expver") if v in dmax.coords])
    dmax.attrs.update(statistic="daily_maximum", sampling="3-hourly (00,03,...,21 UTC)",
                      source="ERA5 hourly SWH, local aggregation")

    dmax.to_netcdf(out_f, encoding={"swh": {"zlib": True, "complevel": 4, "dtype": "float32"}})
    ds.close()
    print(f"[DONE] {year} -> {out_f.name}")
